# Ordinary Differential Equations II

Chapter 9 of the TN23015 book: its explanation and its exercises. Run the first cell before anything else, then work down the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.signal import find_peaks
from tn23015 import check_answers, given, load_data, md, print, row, show, slider

In this lecture, we will explore applying the techniques for solving initial value problems that we learned in [Ordinary Differential Equations I](https://cbl.curve.space/articles/curious-beams-lecture-tn23015-computational-science/ordinary-differential-equations-1). We will also learn how to use numerical integration of ODEs to solve boundary value problems.

> **Learning goals**
>
> 1. Use numerical integration to find the steady-state of the driven damped Harmonic oscillator
> 2. Solve boundary-value problems using the "shooting method"
> 3. Use numerical integration to solve problems with nonlinear damping

## The driven damped Harmonic oscillator

The first physical problem we will be interested in simulating is the driven, damped Harmonic oscillator. 

The physics of a harmonic oscillator is described by the motion of a mass free to move in one dimension that is attached to a rigid wall by a spring that exerts a restoring force:

$$
F_r = -kx
$$

In general, a mass will also experience a friction drag force proportional to and opposing its velocity:

$$
F_f = - c \frac{dx}{dt}
$$

where $c$ is the friction coefficient.
Including a periodic external force $F_{e}(t) = F_0 \cos \omega t$ applied to the mass, the total force becomes:

$$
F_{tot} = - k x - c \frac{dx}{dt} + F_0 \cos \omega t
$$

Using $F = ma$, we find the following second order differential equation for x:

$$
m \frac{d^2x}{dt^2} + c\frac{dx}{dt} + k x  = F_0 \cos(\omega t)
$$

For this equation, there is a [steady-state solution](https://en.wikipedia.org/wiki/Harmonic_oscillator#Steady-state_solution) for $x(t)$ given by:

$$
x(t) = A(\omega) \cos \left[ \omega t + \theta(\omega) \right] 
$$

where $A(\omega)$ and $\theta(\omega)$ are an amplitude and phase that depend on the driving frequency, and are given by:

$$
\begin{aligned}
A(\omega) &= \frac{F_0 / m}{\sqrt{\gamma^2 \omega^2 + (\omega^2 - \omega_0^2)^2}} \\
\theta(\omega) &= -\varphi(\omega), \qquad \tan \varphi(\omega) = \frac{\omega \gamma}{\omega_0^2 - \omega^2}, \quad 0 \leq \varphi \leq \pi
\end{aligned}
$$

where $\omega_0 = \sqrt{k/m}$ is the natural frequency of the harmonic oscillator and $\gamma = c/m$ is the damping rate.
The phase lag $\varphi$ rises from 0 well below resonance, through $\pi/2$ at $\omega = \omega_0$, to $\pi$ well above it.  

In this problem, you will perform numerical integration of the driven damped Harmonic oscillator (DDHO) to find its steady-state response, and compare what you find to the analytical formulas above. 

We will consider the case of $m = 1$ kg, $k = 1$ N/m, $c = 0.1$ Ns/m, and $F_0 = 1$ N.

### Exercise 9.1

Consider the case in which the mass is at rest and is at position $x=0$ at $t=0$, and a driving force at frequency $\omega = \omega_0$.
Use the `solve_ivp()` routine to calculate $x(t)$ for $t$ from 0 to 200 seconds with 1000 points.
Make a plot of your solution and discuss if it behaves in the way that you would expect.

In [ ]:
# Fill these in as you work through the exercise.
_ax = sol = x = None

m = 1
k = 1
c = 0.1
F0 = 1
w = 1

# I will chose y[0] = x and y[1] = v 
def dydt(t,y):
    return None

T = 200
N = 1000
t = np.linspace(0,T,N)

x0 = 0
v0 = 0

# sol = solve_ivp(....)
# t = ...
# x = ...

# Now a plot
# _fig, _ax = plt.subplots()
# ...

# Do not edit or remove the boiler-plate code below.
show(_ax, sol=sol, x=x, dydt=dydt)

In [ ]:
check_answers(x=x, key="answer_12_1a")

If you've done things correctly, you should see an oscillating signal that grows in amplitude and then exponentially approaches a fixed amplitude for long times (this is referred to as the "ring-up" of the oscillator).
At steady state, it looks like the signal is oscillating as we expect.
Qualitatively, it looks good, but does it quantitatively agree with the analytical formulas from above for the steady state behaviour? 

If we want to check this, we need to extract the steady state response from the time trace above.
How do we know after which time our simulated trace has reached steady-state?
Technically, it only approaches steady state exponentially, so it will actually never completely reach steady-state in any case. 

For estimating the steady state amplitude, we could do this by calculating the height of the last maximum of the oscillations.

### Exercise 9.2

Find the steady state amplitude of the calculated time trace by finding the value of the amplitude of the last peak in the trace.
For this, you can use the [`scipy.signal.find_peaks`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.find_peaks.html) routine to find all of the peaks of the oscillations.
Make a plot of the peak values vs. time, and then extract an estimate of the steady state amplitude using the value of the last peak. 

Does it agree with the prediction of the analytical formulas above?
To check this, create a function `amp(w, m, c, k, F0)` (with `w` as $\omega$) to calculate the amplitude from the formula above.
Use this to calculate the theoretically predicted amplitude based on the parameters you have used in the simulation.

In [ ]:
# Fill these in as you work through the exercise.
_ax = amp_calc = amp_theory = t_p = x_p = None

def amp(w, m, c, k, F0):
    return None

# peak_indices, _ = ....
# t_p = ...
# x_p = ...

# A plot
# _fig, _ax = plt.subplots()
# ...

# Extract the calculated amplitude
# amp_calc = ....

# Calculate the theory prediction
# amp_theory = ....

# Do not edit or remove the boiler-plate code below.
if given(amp_calc, amp_theory, t_p, x_p, amp):
    print("Calculated steady state amplitude: ", amp_calc)
    print("Predicted steady state amplitude : ", amp_theory)

show(_ax, amp_calc=amp_calc, amp_theory=amp_theory, t_p=t_p, x_p=x_p, amp=amp)

In [ ]:
check_answers(amp_calc=amp_calc, amp_theory=amp_theory, key="answer_12_1b")

How can we check that the frequency of our steady-state oscillations is correct?
One way we can do this is by using the Fourier transform.

### Exercise 9.3

Calculate the power spectrum of the second half of the DDHO response trace calculated in Exercise 9.1.
Make a plot of the power spectrum with a logarithmic scale on the y-axis, and find the frequency in the data corresponding to the highest spectral power. 

In your power spectrum, keep only the part of the spectrum at positive frequencies.

In [ ]:
# Fill these in as you work through the exercise.
_ax = Tss = f = fmax = fmax_expected = max_index = xss = None

# Pick out the second half of the trace to analyze for the steady state
# xss = ...
# Tss = ...
# power = ...
# f = ...

# Now keep only the positive frequencies
end = int(N/4)

# fmax = ...
# fmax_expected = ...

# Fill in the commented lines below, but leave the rest of this block in place.
f_pow = power = None
if given(Tss, f, fmax, fmax_expected, max_index, xss):
    f_pow = f[0:end]
    power = power[0:end]

    # Make a plot
    # _fig, _ax = plt.subplots()
    # ...

    # Find the frequency of the peak

    print("Calculated frequency:  %.5f Hz (%.5f Rad/s)" %(f[max_index], f[max_index]*2*np.pi))
    print("Theoretical frequency: %.5f Hz (%.5f Rad/s)" % (1/2/np.pi, 1))

show(_ax, Tss=Tss, f=f, fmax=fmax, fmax_expected=fmax_expected, max_index=max_index, xss=xss)

In [ ]:
check_answers(power=power, f_pow=f_pow, fmax=fmax, key="answer_12_1c")

For the simple Harmonic oscillator, the steady state solution does not depend on the initial conditions (this is not true in general for non-linear restoring forces).

### Exercise 9.4

Perform the calculation from Exercise 9.1 with an initial condition $x(0) = 30$.
Make a plot of $x(t)$.
Write a function `find_ss_amp(x)` to find the steady state amplitude as we did in Exercise 9.2 by finding the peak value of the last peak.
Use this function to find the steady state amplitude and show that this is the same (or at least close) to the value for $x(0) = 0$.

In [ ]:
# Fill these in as you work through the exercise.
_ax = sol_2 = t_2 = x_2 = None

def find_ss_amp(x):
    return None

x0_2 = 30
v0_2 = 0
# sol_2 = solve_ivp(....)
# t_2 = ...
# x_2 = ...

# Make a plot
# _fig, _ax = plt.subplots()
# ...

# Do not edit or remove the boiler-plate code below.
if given(sol_2, t_2, x_2, find_ss_amp):
    print("Calculated steady state amplitude: ", find_ss_amp(x_2))

show(_ax, sol=sol_2, t=t_2, x=x_2, find_ss_amp=find_ss_amp)

In [ ]:
check_answers(x_2=x_2, key="answer_12_1d")

## Boundary value problems and the shooting method

Until now, we have been finding the solutions of differential equations given a particular initial condition: for example, for a second order differential equation, we have been specifying the initial velocity and initial position.

There are some times where you want to constrain your solution of the differential equation not by the initial conditions, but by a combination of the initial condition and the final condition.
An example of this is for a second order ODE: instead of finding the solution corresponding to a given initial position $x_i$ and initial velocity $v_i$, one might want to find a solution that has a given initial position $x_i$ and a given final position $x_f$. This is an example of a boundary value problem: we are constrained by the values of the function on the boundaries, not by the initial velocity.

How does one approach such a problem? 

One method for solving this is a technique known as the [shooting method](https://en.wikipedia.org/wiki/Shooting_method).
In the shooting method, one makes a guess at an initial velocity $v_i$ and then calculates the final position $x_f$. If you do not get it right in the first guess, you keep redoing the calculation with different $v_i$, using a technique such as [binary search](https://en.wikipedia.org/wiki/Binary_search_algorithm) to find the initial velocity $v_i$ that gives the desired $x_f$.

_(The inspiration for the name of this technique comes from how one would hit a target with a cannon: a simple way is to take a trial shot, see where it lands, and then adjust the angle up and down until you get it right; or at least close enough to hit your enemy!)_

As an example, we consider the following problem: we are going to throw a ball straight upwards in the air, and we want the ball to land back down at the same spot in exactly 10 seconds.
What velocity do we need to give the ball? 

Neglecting air resistance, this problem is easy to solve (you can solve it with high school physics!).
Once we add air resistance to the problem (which we will at the end of this notebook), the problem becomes very difficult to solve analytically, which is where numerical methods become very useful.

We have learned already how to perform RK4 numerical integration of differential equations, and earlier in the course, we have also learned how to find zeros of a function efficiently using binary search.
With the skills you have learned, you could easily write code to perform the shooting method directly yourself. 

However, here, we will explore how to implement the shooting method using some more advanced features of the [solve_ivp()](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html) routine of the `scipy` library.
To do this, we will make use of the optional parameter `events` of the `solve_ivp()` routine.
This is a special way to tell `solve_ivp()` to detect conditions on the integration it is performing and stop the integration when that condition is satisfied. 

The way the `events=` parameter works is that you need to define a function that gets passed your variables `t` and `y` and which should return a number.
When your "event" function undergoes a zero crossing, then you can have `solve_ivp()` stop the integration automatically.
You can also configure if this zero crossing is a "rising" or a "falling" edge trigger. 

How do we use this `events` parameter in practice?
The `events` parameter needs to take an [object](https://docs.python.org/3/tutorial/classes.html) (or a list of objects in case you want to track many events).

But what is an "object"?

Objects are an advanced feature of Python.
In fact, most of the things you have been working with are already objects, even though you didn't notice it!
Objects are special variables that can "contain" stuff: they can contain, for example, both functions that can perform operations and data that stores values.

An example you have already seen is numpy arrays.
For example, consider the following numpy array:

```
x = np.array(range(10))
```

This array has a "field" (data stored inside the object) called `shape` that tells you what shape the array is:

In [ ]:
# Notebook code
_x = np.array(range(10))
show(_x.shape)

The object `x` also has functions built into it that can execute actions.
For example, there is a function `copy` built into `x` that makes a copy of the array:

```
x2 = x.copy()
```

Cool!
But how do we make our own "object"?
It sounds scary!
In most languages, it requires a more detailed knowledge of the language to do so...
However, python has some great shortcuts that make building your own objects very easy, on the fly, using a technique in python with a funny name called "[monkey patching](https://www.geeksforgeeks.org/monkey-patching-in-python-dynamic-behavior/)".

You can construct the "object" that the `solve_ivp()` parameter `events=` requires by first creating a function that does what you need and then turn it into an "object" by adding some fields with the correct names.
To be concrete, for our example of throwing a ball vertically in the air, we will want to stop the numerical integration once the height of the ball falls back down to zero.
For this, we will need an event function that returns the height of the ball.
If we choose our `y` variable array such that `y[0]` is the vertical position `x` and `y[1]` is the vertical velocity `v`, then my function would look like this:

```
def myevent(t,y):
    return y[0]
```

When you give it this function, the `solve_ivp()` routine will work to find the precise integration conditions and trigger the event at the point where your function `myevent()` returns zero.
In this case, because we are looking for the condition that the height of the ball is zero, then life is very easy: we just return the height of the ball!
If we instead wanted to trigger the event when the height of the ball is 1, then we would replace the last line of our function with `return y[0]-1`. 

For `solve_ivp()`, we then need to make it an object by adding two extra fields.
One field is called `terminal`, which tells `solve_ivp()` if it should stop when this event is triggered, or if it should keep on going.
As we are done once the ball hits the ground, we can make our event terminal:

```
myevent.terminal = True
```

Magico!
Now `myevent` is no longer just a function that can do stuff, but an object that contains a field that stores some information!
In addition, we need to add another field to our `myevent` object that tells `solve_ivp()` if it should look for a "rising edge zero crossing" (direction = 1) or a "falling edge zero crossing" (direction = -1).
We want the latter:

```
myevent.direction = -1
```

When finished, the `sol` object that `solve_ivp()` returns will contain a field `t_events` that is a list of all the times that an event occurred at: for us, this list will contain one entry, corresponding to the time at which the ball came back to the place we threw it from.

With this, it becomes very easy to implement the shooting method:

1. We define an "event" function that returns the value of the height of the ball
2. We make a guess of a lower and upper limit on the velocity in between which we think the correct initial velocity can be found
3. We use binary search to iterate through initial velocities until we find one that gives us a total time of 10 seconds for the ball to fall back down (within a given accuracy). 

One problem with this technique is that we need to get some reasonable values for the initial guesses.
For this, it is handy to run the numerical integration a few times with different values of initial velocity to see what range we should choose for our binary search.

### Exercise 9.5

Consider a ball with mass $m = 0.1$ kg thrown upwards into the air with an initial velocity of 1 m/s.
Use `solve_ivp()` to calculate how long it takes for the ball to reach the ground again.
Assume that $g = 9.81$ m/s$^2$.
For the numerical integration, you need to pick a time endpoint that will be longer than the time it will take for the ball to come back down.
A time limit of 100 seconds should probably be fine.

In [ ]:
# Fill these in as you work through the exercise.
sol_3 = None

m_2=0.1
g = 9.81 

# y[0] is height (which I will call "x"), y[1] is velocity ("v")
def myevent(t,y):
    return None

def dydt_2(t,y):
    return None

vi = 1
# sol_3 = solve_ivp(....)

# Do not edit or remove the boiler-plate code below.
t_f = None
if given(sol_3, myevent, dydt_2):
    t_f = sol_3.t_events[0][0]

    print("An initial velocity of 1 m/s gave a total time of %f seconds" % t_f)

show(sol=sol_3, myevent=myevent, dydt=dydt_2)

In [ ]:
check_answers(t_f=t_f, key="answer_12_2a")

### Exercise 9.6

Implement binary search to find the initial velocity needed to have $t_f = 10$ seconds with a target accuracy of 1 ms.
For your initial search range, choose $v_i$ = 1 m/s and 100 m/s.
Compare to what you would expect theoretically from the acceleration due to gravity with no air resistance.

In [ ]:
# Fill these in as you work through the exercise.
vi_2 = vi_theory = None

def find_tf(vi):
    return None

v1 = 1

# vi_2 = ...
# vi_theory = ...

# Do not edit or remove the boiler-plate code below.
t1 = t2 = target = v2 = None
if given(vi_2, vi_theory, find_tf):
    t1 = find_tf(v1)
    v2 = 200
    t2 = find_tf(v2)
    target = 1e-3

    # while np.abs(t2-t1) > target:

    print(vi_2)
    print(vi_theory)

show(vi=vi_2, vi_theory=vi_theory, find_tf=find_tf)

In [ ]:
check_answers(vi_2=vi_2, vi_theory=vi_theory, key="answer_12_2b")

### Exercise 9.7

We will now add air resistance to our calculation.
To make life simple, we will assume that the [drag coefficient](https://en.wikipedia.org/wiki/Drag_coefficient) of our ball results in a friction force with the magnitude:

$$
|F_f| = Cv^2
$$

For a 3 cm diameter sphere, Gary's estimate of the constant $C$ in this equation (based on the formulas on wikipedia) is that it has a value on the order of $C = 10^{-3}$ Ns$^2$/m$^2$.
Since the force always opposes the direction of the velocity, we have to somehow account for its sign before incorporating it into an equation of motion.
One way to do this is as follows:

$$
F_f = - \frac{Cv^3}{|v|}
$$

Repeat Exercise 9.5 but now including air resistance in your calculation.

Before you start: do you think it will take less time or more time for the ball to fall for the same velocity? 

On the one hand, when falling, the ball will move more slowly because the air resistance is holding it back.
On the other hand, the ball will not go as high with air resistance. 

(Give your `dydt` function with air resistance a different name so we can compare the two in the next question...)

In [ ]:
# Fill these in as you work through the exercise.
sol_4 = None

your_guess = "comes back in less time /OR/ takes more time to come back (delete one!)"

g_2 = 9.81 
m_3 = 0.1
C = 1e-3

# y[0] is height (which I will call "x"), y[1] is velocity ("v")
def myevent_2(t,y):
    return None

def dydt2(t,y):
    return None

vi_3 = 1

# sol_4 = solve_ivp(....)

# Do not edit or remove the boiler-plate code below.
t_f_2 = None
if given(sol_4, myevent_2, dydt2):
    t_f_2 = sol_4.t_events[0][0]

    print("An initial velocity of 1 m/s gave a total time of %f seconds" % t_f_2)

show(sol=sol_4, myevent=myevent_2, dydt2=dydt2)

In [ ]:
check_answers(t_f_2=t_f_2, key="answer_12_2c")

It seems like adding air resistance slightly decreased the amount of time it takes for the ball to come back down, but not by much.

### Exercise 9.8

Make a calculation of the two times as a function of the initial velocity, with $v_i$ varying from 1 m/s to 200 m/s (720 km/h!) with 100 points.
Make a plot of $t_f$ vs $v_i$ for the two cases.

In [ ]:
Ni = 200
vi_4 = np.linspace(1,200,Ni)
tf1 = np.empty(Ni)
tf2 = np.empty(Ni)

# Set `unfinished` to False once you have written the code below, so that it can run.
unfinished = True

if not unfinished:
    for _i in range(Ni):
        ...

        # ... dydt

    for _i in range(Ni):
        ...

        # ... dydt2

    # Make some plots

# Do not edit or remove the boiler-plate code below.
show()

In [ ]:
check_answers(vi_4=vi_4, tf1=tf1, tf2=tf2, key="answer_12_2d", when=not unfinished)

### Exercise 9.9

Calculate the initial velocity required to achieve $t_f$ of 10 seconds including air resistance.

In [ ]:
# Fill these in as you work through the exercise.
vi_5 = None

def find_tf2(vi):
    return None

v1_2 = 1

# vi_5 = ...

# Do not edit or remove the boiler-plate code below.
t1_2 = t2_2 = target_2 = v2_2 = None
if given(vi_5, find_tf2):
    t1_2 = find_tf2(v1_2)
    v2_2 = 200
    t2_2 = find_tf2(v2_2)
    target_2 = 1e-3

    # while np.abs(...) > target:

    print(vi_5)

show(vi=vi_5, find_tf2=find_tf2)

In [ ]:
check_answers(vi_5=vi_5, key="answer_12_2e")

### Exercise 9.10

Including air resistance, does the ball spend more time on the upwards trajectory, on the downward trajectory, or does it spend the same amount of time going up as it does going down?

Use the `solve_ivp()` routine to calculate the time going up and the time going down for the initial velocity you found in Exercise 9.9. 

_Hint:_ an easy way to do this is to create an additional `terminal = False` event function that checks when the velocity crosses zero!

In [ ]:
# Fill these in as you work through the exercise.
sol_5 = None

# y[0] is height (which I will call "x"), y[1] is velocity ("v")
def myevent_3(t,y):
    return None

def myevent2(t,y):
    return None

# sol_5 = solve_ivp(..., events=[myevent2, myevent])

# Do not edit or remove the boiler-plate code below.
t1_3 = t2_3 = t_down = None
if given(sol_5, myevent_3, myevent2):
    t1_3 = sol_5.t_events[0][0]
    t2_3 = sol_5.t_events[1][0]
    t_down = t2_3-t1_3

    print("Time up:   %f" % t1_3)
    print("Time down: %f" % (t2_3-t1_3))

show(sol=sol_5, myevent=myevent_3, myevent2=myevent2)

In [ ]:
check_answers(t1_3=t1_3, t_down=t_down, key="answer_12_2f")